# Chains
* Perform several actions in a particular order.

## Setup

#### After you download the code from the github repository in your computer
In terminal:
* cd project_name
* pyenv local 3.11.4
* poetry install
* poetry shell

#### To open the notebook with Jupyter Notebooks
In terminal:
* jupyter lab

Go to the folder of notebooks and open the right notebook.

#### To see the code in Virtual Studio Code or your editor of choice.
* open Virtual Studio Code or your editor of choice.
* open the project-folder
* open the 004-chains.py file

## Create your .env file
* In the github repo we have included a file named .env.example
* Rename that file to .env file and here is where you will add your confidential api keys. Remember to include:
* OPENAI_API_KEY=your_openai_api_key
* LANGCHAIN_TRACING_V2=true
* LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
* LANGCHAIN_API_KEY=your_langchain_api_key
* LANGCHAIN_PROJECT=your_project_name

We will call our LangSmith project **004-chains**.

## Track operations
From now on, we can track the operations **and the cost** of this project from LangSmith:
* [smith.langchain.com](https://smith.langchain.com)

## Connect with the .env file located in the same directory of this notebook

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [ ]:
#pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

#### Install LangChain

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [ ]:
#!pip install langchain

## Connect with an LLM

If you are using the pre-loaded poetry shell, you do not need to install the following package because it is already pre-loaded for you:

In [ ]:
#!pip install langchain-openai

* NOTE: Since right now is the best LLM in the market, we will use OpenAI by default. You will see how to connect with other Open Source LLMs like Llama3 or Mistral in a next lesson.

## LLM Model
* The trend before the launch of chatGPT-4.
* See LangChain documentation about LLM Models [here](https://python.langchain.com/v0.1/docs/modules/model_io/llms/).

#### Update note: current OpenAI models
* The original example is very good for learning the concept, and it worked with the OpenAI models available when the course was recorded: `OpenAI()` used the text-completion model `gpt-3.5-turbo-instruct`, and `ChatOpenAI` used `gpt-3.5-turbo-0125`.
* OpenAI has since retired both models. With the current models, we would do it like this: a chat model through `ChatOpenAI`, with `gpt-6-luna` (small, fast and affordable).
* `llmModel` keeps its name so the rest of the notebook works the same way, but it is now a chat model: it returns a message, and the text is in `response.content`.
* The version of `langchain-openai` used in this project sends `temperature=0.7` by default, but recent OpenAI models only accept the default value (1). That is why we set `temperature=1` explicitly.
* To check which models are available at any time, see OpenAI's deprecations page: [developers.openai.com/api/docs/deprecations](https://developers.openai.com/api/docs/deprecations).

In [ ]:
from langchain_openai import ChatOpenAI

MODEL = "gpt-6-luna"  # small, fast and affordable current OpenAI model

# This version of langchain-openai sends temperature=0.7 by default,
# but recent OpenAI models only accept the default value (1).
llmModel = ChatOpenAI(model=MODEL, temperature=1)

## Chat Model
* The general trend after the launch of chatGPT-4.
    * Frequently known as "Chatbot". 
    * Conversation between Human and AI.
    * Can have a system prompt defining the tone or the role of the AI. 
* See LangChain documentation about Chat Models [here](https://python.langchain.com/v0.1/docs/modules/model_io/chat/).
* By default we will work with ChatOpenAI. See [here](https://python.langchain.com/v0.1/docs/integrations/chat/openai/) the LangChain documentation page about it.

In [ ]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model=MODEL, temperature=1)

## Prompts
* See the LangChain documentation about prompts [here](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/quick_start/).
* Input into LLMs.
* Prompt templates: easier to use prompts with variables. A prompt template may include:
    * instructions,
    * few-shot examples,
    * and specific context and questions appropriate for a given task.

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}."
)

llmModelPrompt = prompt_template.format(
    adjective="curious", 
    topic="the Kennedy family"
)

llmModel.invoke(llmModelPrompt).content

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an {profession} expert on {topic}."),
        ("human", "Hello, Mr. {profession}, can you please answer a question?"),
        ("ai", "Sure!"),
        ("human", "{user_input}"),
    ]
)

messages = chat_template.format_messages(
    profession="Historian",
    topic="The Kennedy family",
    user_input="How many grandchildren had Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)

In [ ]:
response

In [ ]:
print(response)

In [ ]:
print(response.content)

#### Old way:

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content=(
                "You are an Historian expert on the Kennedy family."
            )
        ),
        HumanMessagePromptTemplate.from_template("{user_input}"),
    ]
)

messages = chat_template.format_messages(
    user_input="Name the children and grandchildren of Joseph P. Kennedy?"
)

response = chatModel.invoke(messages)

In [ ]:
print(response.content)

#### What is the full potential of ChatPromptTemplate?
* Check the [corresponding page](https://api.python.langchain.com/en/latest/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html) in the LangChain API.

## Our first chain: an example of few-shot prompting

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

In [ ]:
examples = [
    {"input": "hi!", "output": "¡hola!"},
    {"input": "bye!", "output": "¡adiós!"},
]

In [ ]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an English-Spanish translator."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

chain = final_prompt | chatModel

chain.invoke({"input": "Who was JFK?"})

## How to execute the code from Visual Studio Code
* In Visual Studio Code, see the file 001-connect-llms.py
* In terminal, make sure you are in the directory of the file and run:
    * python 004-chains.py